# DipRadar ML v1.0 — Modelo Principal de Produção

**Objectivo:** Treinar o modelo definitivo XGB-v2 (30 features) para scoring de dips em produção.

**Arquitectura:**
- `model_up`   → prevê `max_return_60d`  (upside potencial, 60 dias)
- `model_down` → prevê `max_drawdown_60d` (risco máximo de queda, 60 dias)
- `score = pred_up / (|pred_down| + ε)` → rácio risco/retorno

**Validação:** Walk-Forward CV, 5 folds, TimeSeriesSplit  
**Dataset:** `ml_training_merged.parquet` (raiz do repo)

## 0. Setup

In [ ]:
# Clonar repo (Colab)
import os
if not os.path.exists('/content/DipRadar'):
    !git clone https://github.com/romeurf/DipRadar.git /content/DipRadar
else:
    !git -C /content/DipRadar pull --quiet

!pip install xgboost scikit-learn pandas numpy scipy joblib pyarrow --quiet
print('✅ Setup concluído.')

## 1. Imports & Config

In [ ]:
import sys, json, warnings
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from datetime import datetime
from scipy.stats import spearmanr, mstats
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)

REPO        = Path('/content/DipRadar')
DATA_PATH   = REPO / 'ml_training_merged.parquet'
MODEL_DIR   = REPO / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TARGET_UP   = 'max_return_60d'
TARGET_DOWN = 'max_drawdown_60d'
N_FOLDS     = 5

# Feature set completo v2 (30 features)
FEATURES_WANTED = [
    # macro
    'macro_score', 'vix', 'vix_regime', 'sp500_trend',
    # dip
    'drop_pct_today', 'drawdown_52w', 'distance_from_ma50',
    # momentum
    'return_5d', 'return_10d', 'return_20d', 'return_1m', 'return_3m_pre',
    # mean-reversion / volatilidade
    'zscore_20d', 'atr_ratio', 'volatility_20d', 'beta_60d',
    # volume
    'volume_spike',
    # contexto mercado
    'spy_drawdown_5d', 'sector_drawdown_5d', 'sector_relative',
    # fundamentais
    'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside',
    'quality_score', 'pe_attractive',
    # engineered
    'drop_x_drawdown', 'vol_x_drop',
    # RSI
    'rsi_14', 'rsi_oversold_strength',
]

print(f'Features desejadas: {len(FEATURES_WANTED)}')
print(f'DATA_PATH: {DATA_PATH}')
print(f'MODEL_DIR: {MODEL_DIR}')

## 2. Carregar Dataset & Diagnóstico

In [ ]:
df = pd.read_parquet(DATA_PATH)

# Normalizar coluna de data
date_col = next((c for c in ['alert_date','date_iso','date'] if c in df.columns), None)
assert date_col, 'Nenhuma coluna de data encontrada!'
df['alert_date'] = pd.to_datetime(df[date_col], errors='coerce')
df = df.sort_values('alert_date').reset_index(drop=True)

print('=== DIAGNÓSTICO DO DATASET ===')
print(f'Shape          : {df.shape}')
print(f'Janela temporal: {df["alert_date"].min().date()} → {df["alert_date"].max().date()}')
print(f'Todas as colunas ({len(df.columns)}): {sorted(df.columns.tolist())}')

# Features disponíveis vs em falta
FEATURES = [f for f in FEATURES_WANTED if f in df.columns]
missing_feats = [f for f in FEATURES_WANTED if f not in df.columns]
print(f'\nFeatures activas : {len(FEATURES)}')
if missing_feats:
    print(f'Features em falta: {missing_feats}')
    print('⚠️  Serão usadas 0.0 como fallback para features em falta.')
    for f in missing_feats:
        df[f] = 0.0
    FEATURES = FEATURES_WANTED  # usar todas com fallback

# Verificar targets
print(f'\nTargets presentes:')
print(f'  {TARGET_UP}   : {TARGET_UP in df.columns}')
print(f'  {TARGET_DOWN} : {TARGET_DOWN in df.columns}')

## 2b. Calcular Targets (se ausentes)

In [ ]:
# Se os targets não existirem no parquet, calculá-los via yfinance
if TARGET_UP not in df.columns or TARGET_DOWN not in df.columns:
    print('⚠️  Targets ausentes — a calcular via yfinance...')
    import yfinance as yf
    import time

    sys.path.insert(0, str(REPO))
    from experiments.ml_v2.pipeline import build_targets

    # Identificar coluna de ticker e preço
    ticker_col = next((c for c in ['ticker','symbol'] if c in df.columns), None)
    price_col  = next((c for c in ['entry_price','price','price_alert','close'] if c in df.columns), None)
    assert ticker_col and price_col, f'Colunas em falta: ticker={ticker_col}, price={price_col}'

    df['ticker']      = df[ticker_col].str.strip().str.upper()
    df['entry_price'] = pd.to_numeric(df[price_col], errors='coerce')

    results = []
    for ticker, grp in df.groupby('ticker'):
        earliest = grp['alert_date'].min()
        latest   = grp['alert_date'].max()
        start    = (earliest - pd.Timedelta(days=5)).strftime('%Y-%m-%d')
        end      = (latest + pd.Timedelta(days=70)).strftime('%Y-%m-%d')
        try:
            ohlcv = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if ohlcv.index.tz is not None:
                ohlcv.index = ohlcv.index.tz_localize(None)
            for idx, row in grp.iterrows():
                alert_date  = pd.Timestamp(row['alert_date'])
                entry_price = float(row['entry_price']) if pd.notna(row['entry_price']) else 0
                future = ohlcv[
                    (ohlcv.index > alert_date) &
                    (ohlcv.index <= alert_date + pd.Timedelta(days=60))
                ]['Close']
                t = build_targets(alert_date, entry_price, future, horizon_days=60)
                results.append({'_idx': idx, **t})
        except Exception as e:
            for idx in grp.index:
                results.append({'_idx': idx, 'max_return_60d': np.nan,
                                'max_drawdown_60d': np.nan, 'min_idx': -1})
        time.sleep(0.3)

    df_tgt = pd.DataFrame(results).set_index('_idx')
    df[TARGET_UP]   = df_tgt['max_return_60d']
    df[TARGET_DOWN] = df_tgt['max_drawdown_60d']
    print(f'✅ Targets calculados: {df[[TARGET_UP, TARGET_DOWN]].notna().sum().to_dict()}')
else:
    print('✅ Targets já presentes no dataset.')

# Remover linhas sem targets
n_before = len(df)
df = df.dropna(subset=[TARGET_UP, TARGET_DOWN]).reset_index(drop=True)
print(f'Amostras com targets válidos: {len(df):,} (removidas {n_before - len(df):,})')

print(f'\n--- Distribuição dos Targets ---')
print(df[[TARGET_UP, TARGET_DOWN]].describe().round(4))

## 3. Funções Auxiliares

In [ ]:
def winsorize_col(arr, limits=(0.01, 0.01)):
    return np.array(mstats.winsorize(arr, limits=limits))

def log1p_signed(x):
    return np.sign(x) * np.log1p(np.abs(x))

def inv_log1p_signed(x):
    return np.sign(x) * np.expm1(np.abs(x))

def transform_targets(y_up, y_down):
    return log1p_signed(y_up), log1p_signed(y_down)

def inverse_transform(y_up_t, y_down_t):
    return inv_log1p_signed(y_up_t), inv_log1p_signed(y_down_t)

def compute_score(pred_up, pred_down, eps=0.01):
    return pred_up / (np.abs(pred_down) + eps)

def spearman_metrics(pred_up, y_up, pred_down, y_down):
    rho_up,   _ = spearmanr(pred_up,   y_up)
    rho_down, _ = spearmanr(pred_down, y_down)
    return {
        'rho_up':   round(float(rho_up),   4),
        'rho_down': round(float(rho_down), 4),
        'rho_mean': round(float((rho_up + rho_down) / 2), 4),
    }

def topk_pnl(pred_up, pred_down, y_up, k=0.20):
    scores = compute_score(pred_up, pred_down)
    n = max(1, int(len(scores) * k))
    idx = np.argsort(scores)[-n:]
    return round(float(np.mean(y_up[idx])), 4)

def bottomk_pnl(pred_up, pred_down, y_up, k=0.20):
    scores = compute_score(pred_up, pred_down)
    n = max(1, int(len(scores) * k))
    idx = np.argsort(scores)[:n]
    return round(float(np.mean(y_up[idx])), 4)

def make_model_up():
    return XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=0
    )

def make_model_down():
    return XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.2, reg_lambda=2.0,
        random_state=42, n_jobs=-1, verbosity=0
    )

print('✅ Funções definidas.')

## 4. Walk-Forward CV (Validação OOS)

In [ ]:
TEST_SIZE = max(200, int(len(df) * 0.12))
tscv = TimeSeriesSplit(n_splits=N_FOLDS, test_size=TEST_SIZE)

cv_results = []
print(f'Walk-Forward CV — {N_FOLDS} folds, test_size={TEST_SIZE}\n')

for k, (tr_idx, te_idx) in enumerate(tscv.split(df)):
    df_tr = df.iloc[tr_idx]
    df_te = df.iloc[te_idx]

    X_tr = df_tr[FEATURES].fillna(0).values.astype(np.float32)
    X_te = df_te[FEATURES].fillna(0).values.astype(np.float32)

    y_up_tr   = winsorize_col(df_tr[TARGET_UP].fillna(0).values)
    y_down_tr = winsorize_col(df_tr[TARGET_DOWN].fillna(0).values)
    y_up_tr_t, y_down_tr_t = transform_targets(y_up_tr, y_down_tr)

    y_up_te   = df_te[TARGET_UP].fillna(0).values
    y_down_te = df_te[TARGET_DOWN].fillna(0).values

    m_up   = make_model_up();   m_up.fit(X_tr,   y_up_tr_t)
    m_down = make_model_down(); m_down.fit(X_tr, y_down_tr_t)

    pred_up, pred_down = inverse_transform(m_up.predict(X_te), m_down.predict(X_te))

    metrics  = spearman_metrics(pred_up, y_up_te, pred_down, y_down_te)
    pnl_top  = topk_pnl(pred_up, pred_down, y_up_te)
    pnl_bot  = bottomk_pnl(pred_up, pred_down, y_up_te)

    te_dates = df['alert_date'].iloc[te_idx]
    row = {
        'fold': k+1, 'n_train': len(tr_idx), 'n_test': len(te_idx),
        'date_start': te_dates.min().date(), 'date_end': te_dates.max().date(),
        **metrics, 'topk_pnl': pnl_top, 'bottomk_pnl': pnl_bot,
    }
    cv_results.append(row)

    print(f'  Fold {k+1} | {te_dates.min().date()} → {te_dates.max().date()} | '
          f'rho_up={metrics["rho_up"]:.4f} rho_down={metrics["rho_down"]:.4f} '
          f'rho_mean={metrics["rho_mean"]:.4f} | '
          f'top20%={pnl_top:.4f} bot20%={pnl_bot:.4f}')

df_cv = pd.DataFrame(cv_results)

print('\n=== Média Walk-Forward CV ===')
avg = df_cv[['rho_up','rho_down','rho_mean','topk_pnl','bottomk_pnl']].mean().round(4)
for k, v in avg.items():
    print(f'  {k:20s}: {v}')

# Critérios de validação
print('\n=== Critérios de Validação ===')
print(f'  rho_mean >= 0.10  : {"✅ PASS" if avg["rho_mean"] >= 0.10 else "❌ FAIL"} ({avg["rho_mean"]:.4f})')
print(f'  topk_pnl >= 0.05  : {"✅ PASS" if avg["topk_pnl"] >= 0.05 else "❌ FAIL"} ({avg["topk_pnl"]:.4f})')
print(f'  topk > bottomk    : {"✅ PASS" if avg["topk_pnl"] > avg["bottomk_pnl"] else "❌ FAIL"}')

## 5. Treino Final (Todo o Dataset)

In [ ]:
X_all      = df[FEATURES].fillna(0).values.astype(np.float32)
y_up_all   = winsorize_col(df[TARGET_UP].fillna(0).values)
y_down_all = winsorize_col(df[TARGET_DOWN].fillna(0).values)
y_up_t, y_down_t = transform_targets(y_up_all, y_down_all)

print(f'A treinar model_up   ({len(X_all):,} amostras)...')
final_model_up = make_model_up()
final_model_up.fit(X_all, y_up_t)

print(f'A treinar model_down ({len(X_all):,} amostras)...')
final_model_down = make_model_down()
final_model_down.fit(X_all, y_down_t)

print('✅ Treino final concluído.')

## 6. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, model, label, color in [
    (axes[0], final_model_up,   'model_up (upside)',  'steelblue'),
    (axes[1], final_model_down, 'model_down (risco)', 'tomato'),
]:
    fi = pd.Series(model.feature_importances_, index=FEATURES)
    fi.sort_values(ascending=True).tail(20).plot(kind='barh', ax=ax, color=color)
    ax.set_title(f'Top-20 Feature Importance — {label}')
    ax.set_xlabel('Importance')

plt.tight_layout()
fig.savefig(MODEL_DIR / 'feature_importance_v1.png', dpi=120)
plt.show()
print(f'Guardado: {MODEL_DIR}/feature_importance_v1.png')

## 7. Guardar Modelos & Metadata

In [ ]:
joblib.dump(final_model_up,   MODEL_DIR / 'model_up_v1.joblib')
joblib.dump(final_model_down, MODEL_DIR / 'model_down_v1.joblib')

with open(MODEL_DIR / 'features_v1.json', 'w') as f:
    json.dump(FEATURES, f, indent=2)

avg = df_cv[['rho_up','rho_down','rho_mean','topk_pnl','bottomk_pnl']].mean().round(4)

meta = {
    'version':    'v1.0',
    'trained_on': datetime.now().isoformat(),
    'n_samples':  len(df),
    'date_range': [str(df['alert_date'].min().date()), str(df['alert_date'].max().date())],
    'n_features': len(FEATURES),
    'features':   FEATURES,
    'targets':    {'up': TARGET_UP, 'down': TARGET_DOWN},
    'cv_folds':   N_FOLDS,
    'cv_results': cv_results,
    'cv_mean': {
        'rho_up':      float(avg['rho_up']),
        'rho_down':    float(avg['rho_down']),
        'rho_mean':    float(avg['rho_mean']),
        'topk_pnl':    float(avg['topk_pnl']),
        'bottomk_pnl': float(avg['bottomk_pnl']),
    },
    'thresholds': {'strong_buy': 2.0, 'buy': 1.5, 'avoid': 0.5},
    'notes': 'XGB-v2, walk-forward 5 folds. Re-treinar trimestralmente ou quando rho_mean < 0.10.'
}

with open(MODEL_DIR / 'metadata_v1.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)

print(f'✅ Modelos guardados em: {MODEL_DIR}')
for p in sorted(MODEL_DIR.iterdir()):
    print(f'  {p.name:40s} {p.stat().st_size/1024:.1f} KB')

## 8. Exemplo de Inferência (Produção)

In [ ]:
def score_alert(alert_features: dict) -> dict:
    """Scoring de um alerta individual. Usa modelos guardados em MODEL_DIR."""
    m_up   = joblib.load(MODEL_DIR / 'model_up_v1.joblib')
    m_down = joblib.load(MODEL_DIR / 'model_down_v1.joblib')
    feats  = json.load(open(MODEL_DIR / 'features_v1.json'))

    x = np.array([[alert_features.get(f, 0.0) for f in feats]], dtype=np.float32)

    pred_up_t   = m_up.predict(x)[0]
    pred_down_t = m_down.predict(x)[0]
    pred_up, pred_down = inv_log1p_signed(pred_up_t), inv_log1p_signed(pred_down_t)

    score = float(compute_score(np.array([pred_up]), np.array([pred_down]))[0])

    if score >= 2.0:   signal = 'STRONG BUY'
    elif score >= 1.5: signal = 'BUY'
    elif score < 0.5:  signal = 'AVOID'
    else:              signal = 'NEUTRAL'

    return {
        'pred_up':   round(float(pred_up),   4),
        'pred_down': round(float(pred_down), 4),
        'score':     round(score, 4),
        'signal':    signal,
    }

# Teste com valores dummy
dummy = {f: 0.0 for f in FEATURES}
dummy.update({
    'atr_ratio':       0.05,
    'vix':             28.0,
    'drawdown_52w':   -0.35,
    'drop_pct_today': -0.08,
    'drop_x_drawdown': 0.028,
    'rsi_14':          30.0,
    'macro_score':      0.6,
})
result = score_alert(dummy)
print('Exemplo de scoring:')
for k, v in result.items():
    print(f'  {k:12s}: {v}')

## 9. Notas de Produção

### Thresholds de scoring
```
score >= 2.0  → STRONG BUY
score >= 1.5  → BUY
score <  0.5  → AVOID
```

### Critérios para manter em produção
- `rho_mean >= 0.10` nos últimos 6 meses de OOS
- `topk_pnl > bottomk_pnl` (modelo discrimina)
- Performance estável fora do período COVID (2022-2026)

### Re-treino
- **Frequência:** trimestral (ou quando `rho_mean` cair abaixo de 0.10)
- **Processo:** expanding window — acrescentar novos alertas ao `ml_training_merged.parquet`
- **Promoção:** comparar `rho_mean` e `topk_pnl` vs v1.0 antes de substituir

### Próximos passos
1. Correr `DipRadar_v1_Backtest.ipynb` para validação com TP/SL
2. Fix `sp500_trend` — actualmente 0.0 constante
3. Melhorar `model_down` — investigar distribuição de `max_drawdown_60d`
4. Regime split: bull vs bear, high-VIX vs low-VIX
5. Integrar `score_alert()` no `ml_predictor.py` do bot